**Cell 1: Instalasi dependency**

In [ ]:
# Library CRF untuk sequence labeling
%pip install -q pytorch-crf==0.7.2

print("✓ Cell 1 berhasil: pytorch-crf sudah terpasang.")

**Cell 2: Import library dan konfigurasi perangkat**

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from google.colab import drive
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF
from tqdm.auto import tqdm

drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch :", torch.__version__)
print("Device  :", DEVICE)
print("✓ Cell 2 berhasil dijalankan.")

**Cell 3: Path dan konfigurasi evaluasi**

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/BiLSTM-CRF"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAPPING_DIR = PROJECT_ROOT / "mappings"

BASELINE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "baseline"
    / "best_model.pt"
)

FINETUNED_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "best_finetuned_model.pt"
)

TEST_PATH = PROCESSED_DIR / "fathul_test.csv"

CHAR2ID_PATH = MAPPING_DIR / "char2id.json"
LABEL2ID_PATH = MAPPING_DIR / "label2id.json"

OUTPUT_DIR = (
    PROJECT_ROOT / "evaluation" / "comparison"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_CSV_PATH = (
    OUTPUT_DIR / "metrics_comparison.csv"
)

METRICS_JSON_PATH = (
    OUTPUT_DIR / "metrics_comparison.json"
)

BASELINE_PREDICTIONS_PATH = (
    OUTPUT_DIR / "baseline_predictions.csv"
)

FINETUNED_PREDICTIONS_PATH = (
    OUTPUT_DIR / "finetuned_predictions.csv"
)

MAX_LEN = 300
BATCH_SIZE = 32
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.30

required_files = [
    TEST_PATH,
    CHAR2ID_PATH,
    LABEL2ID_PATH,
    BASELINE_MODEL_PATH,
    FINETUNED_MODEL_PATH,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {file_path}"
        )

print("Test data       :", TEST_PATH)
print("Baseline model  :", BASELINE_MODEL_PATH)
print("Fine-tuned model:", FINETUNED_MODEL_PATH)
print("Evaluation output:", OUTPUT_DIR)
print("✓ Cell 3 berhasil dijalankan.")

**Cell 4: Memuat test set dan mapping**

In [ ]:
test_data = pd.read_csv(
    TEST_PATH,
    encoding="utf-8-sig",
)

with open(
    CHAR2ID_PATH,
    encoding="utf-8",
) as file:
    char2id = json.load(file)

with open(
    LABEL2ID_PATH,
    encoding="utf-8",
) as file:
    label2id = json.load(file)

with open(
    PROCESSED_DIR / "fathul_split_pages.json",
    encoding="utf-8",
) as file:
    split_pages = json.load(file)

id2char = {
    index: char
    for char, index in char2id.items()
}

id2label = {
    index: label
    for label, index in label2id.items()
}

required_columns = {
    "page",
    "chunk_id",
    "text_undiacritized",
    "text_diacritized",
    "length",
    "split",
}

assert required_columns.issubset(test_data.columns)
assert test_data["split"].eq("test").all()
assert test_data["length"].between(1, MAX_LEN).all()
assert test_data["chunk_id"].is_unique

expected_test_pages = set(
    split_pages["test"]
)

actual_test_pages = set(
    map(int, test_data["page"].unique())
)

assert actual_test_pages == expected_test_pages
assert len(actual_test_pages) == 33
assert len(label2id) == 15
assert len(char2id) == 55

PAD_CHAR_ID = char2id["<PAD>"]
UNK_CHAR_ID = char2id["<UNK>"]
O_LABEL_ID = label2id["O"]

test_summary = pd.DataFrame([
    {
        "split": "test",
        "pages": test_data["page"].nunique(),
        "sequences": len(test_data),
        "characters": int(test_data["length"].sum()),
        "minimum_length": int(
            test_data["length"].min()
        ),
        "maximum_length": int(
            test_data["length"].max()
        ),
    }
])

display(test_summary)

print("Vocabulary size :", len(char2id))
print("Number of labels:", len(label2id))
print("Test set berhasil dibuka untuk evaluasi.")
print("✓ Cell 4 berhasil dijalankan.")

**Cell 5: Dataset dan DataLoader test**

In [ ]:
HARAKAT = set("ًٌٍَُِّْ")


def extract_chars_and_labels(text):
    chars = []
    labels = []
    index = 0

    while index < len(text):
        char = text[index]

        if char in HARAKAT:
            raise ValueError(
                f"Harakat tanpa karakter dasar pada indeks {index}."
            )

        chars.append(char)
        marks = []
        next_index = index + 1

        while (
            next_index < len(text)
            and text[next_index] in HARAKAT
        ):
            mark = text[next_index]

            if mark not in marks:
                marks.append(mark)

            next_index += 1

        label = "|".join(marks) if marks else "O"

        if label not in label2id:
            raise ValueError(
                f"Label tidak tersedia: {repr(label)}"
            )

        labels.append(label)
        index = next_index

    return chars, labels


class DiacritizationDataset(Dataset):
    def __init__(self, frame):
        self.samples = []
        self.unknown_characters = Counter()
        self.label_distribution = Counter()

        for row in frame.itertuples(index=False):
            chars, labels = extract_chars_and_labels(
                row.text_diacritized
            )

            assert len(chars) == len(labels)
            assert "".join(chars) == row.text_undiacritized
            assert len(chars) == int(row.length)

            self.unknown_characters.update(
                char
                for char in chars
                if char not in char2id
            )

            self.label_distribution.update(labels)

            char_ids = [
                char2id.get(char, UNK_CHAR_ID)
                for char in chars
            ]

            label_ids = [
                label2id[label]
                for label in labels
            ]

            self.samples.append({
                "char_ids": char_ids,
                "label_ids": label_ids,
                "chars": chars,
                "labels": labels,
                "length": len(chars),
                "page": int(row.page),
                "chunk_id": row.chunk_id,
                "text_undiacritized": row.text_undiacritized,
                "text_diacritized": row.text_diacritized,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def collate_batch(batch):
    lengths = [
        sample["length"]
        for sample in batch
    ]

    max_batch_length = max(lengths)

    char_batch = []
    label_batch = []
    mask_batch = []

    for sample in batch:
        padding_length = (
            max_batch_length - sample["length"]
        )

        char_batch.append(
            sample["char_ids"]
            + [PAD_CHAR_ID] * padding_length
        )

        label_batch.append(
            sample["label_ids"]
            + [O_LABEL_ID] * padding_length
        )

        mask_batch.append(
            [True] * sample["length"]
            + [False] * padding_length
        )

    return {
        "char_ids": torch.tensor(
            char_batch,
            dtype=torch.long,
        ),
        "label_ids": torch.tensor(
            label_batch,
            dtype=torch.long,
        ),
        "mask": torch.tensor(
            mask_batch,
            dtype=torch.bool,
        ),
        "lengths": torch.tensor(
            lengths,
            dtype=torch.long,
        ),
        "samples": batch,
    }


test_dataset = DiacritizationDataset(test_data)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

assert len(test_dataset) == len(test_data)

sample_batch = next(iter(test_loader))

assert sample_batch["char_ids"].shape == (
    sample_batch["label_ids"].shape
)

assert sample_batch["char_ids"].shape == (
    sample_batch["mask"].shape
)

assert sample_batch["mask"][:, 0].all()

print("Jumlah sequence test :", len(test_dataset))
print("Jumlah batch test    :", len(test_loader))
print("Bentuk batch contoh  :", sample_batch["char_ids"].shape)
print(
    "Unknown test         :",
    dict(test_dataset.unknown_characters),
)
print(
    "Label muncul di test :",
    len(test_dataset.label_distribution),
    "dari",
    len(label2id),
)
print("✓ Cell 5 berhasil dijalankan.")

**Cell 6: Arsitektur BiLSTM-CRF**

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        pad_char_id,
    ):
        super().__init__()

        if hidden_dim % 2 != 0:
            raise ValueError(
                "Hidden dimension harus bernilai genap."
            )

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_char_id,
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        self.output_dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim,
            num_labels,
        )

        self.crf = CRF(
            num_tags=num_labels,
            batch_first=True,
        )

    def compute_emissions(
        self,
        char_ids,
        lengths,
    ):
        embedded = self.embedding(char_ids)
        embedded = self.embedding_dropout(embedded)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.bilstm(packed)

        encoded, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=char_ids.size(1),
        )

        encoded = self.output_dropout(encoded)

        return self.classifier(encoded)

    def loss(
        self,
        char_ids,
        label_ids,
        mask,
        lengths,
    ):
        emissions = self.compute_emissions(
            char_ids,
            lengths,
        )

        log_likelihood = self.crf(
            emissions,
            label_ids,
            mask=mask,
            reduction="token_mean",
        )

        return -log_likelihood

    def decode(
        self,
        char_ids,
        mask,
        lengths,
    ):
        emissions = self.compute_emissions(
            char_ids,
            lengths,
        )

        return self.crf.decode(
            emissions,
            mask=mask,
        )


def create_model():
    return BiLSTMCRF(
        vocab_size=len(char2id),
        num_labels=len(label2id),
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pad_char_id=PAD_CHAR_ID,
    ).to(DEVICE)


baseline_model = create_model()
finetuned_model = create_model()

baseline_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

finetuned_parameters = sum(
    parameter.numel()
    for parameter in finetuned_model.parameters()
    if parameter.requires_grad
)

assert baseline_parameters == finetuned_parameters
assert baseline_parameters == 275342

print("Parameter baseline   :", f"{baseline_parameters:,}")
print("Parameter fine-tuned :", f"{finetuned_parameters:,}")
print("Arsitektur kedua model identik.")
print("✓ Cell 6 berhasil dijalankan.")

**Cell 7: Memuat checkpoint kedua model**

In [ ]:
baseline_checkpoint = torch.load(
    BASELINE_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

finetuned_checkpoint = torch.load(
    FINETUNED_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)


required_checkpoint_keys = {
    "epoch",
    "model_state_dict",
    "validation_loss",
    "char2id",
    "label2id",
}

assert required_checkpoint_keys.issubset(
    baseline_checkpoint.keys()
)

assert required_checkpoint_keys.issubset(
    finetuned_checkpoint.keys()
)


# Memastikan mapping kedua checkpoint identik
assert baseline_checkpoint["char2id"] == char2id
assert baseline_checkpoint["label2id"] == label2id

assert finetuned_checkpoint["char2id"] == char2id
assert finetuned_checkpoint["label2id"] == label2id


# Memuat bobot model
baseline_load_result = baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True,
)

finetuned_load_result = finetuned_model.load_state_dict(
    finetuned_checkpoint["model_state_dict"],
    strict=True,
)

assert not baseline_load_result.missing_keys
assert not baseline_load_result.unexpected_keys

assert not finetuned_load_result.missing_keys
assert not finetuned_load_result.unexpected_keys


BASELINE_EPOCH = int(
    baseline_checkpoint["epoch"]
)

FINETUNED_EPOCH = int(
    finetuned_checkpoint["epoch"]
)

BASELINE_VALIDATION_LOSS = float(
    baseline_checkpoint["validation_loss"]
)

FINETUNED_VALIDATION_LOSS = float(
    finetuned_checkpoint["validation_loss"]
)


# Memastikan checkpoint terbaik yang digunakan
assert BASELINE_EPOCH == 78
assert FINETUNED_EPOCH == 46

assert (
    finetuned_checkpoint["source_dataset"]
    == "Tashkeela -> Fathul Qorib"
)

assert int(
    finetuned_checkpoint["pretrained_epoch"]
) == 62


baseline_model.to(DEVICE)
finetuned_model.to(DEVICE)

baseline_model.eval()
finetuned_model.eval()


# Menonaktifkan gradient karena hanya evaluasi
for parameter in baseline_model.parameters():
    parameter.requires_grad_(False)

for parameter in finetuned_model.parameters():
    parameter.requires_grad_(False)


assert not baseline_model.training
assert not finetuned_model.training

assert not any(
    parameter.requires_grad
    for parameter in baseline_model.parameters()
)

assert not any(
    parameter.requires_grad
    for parameter in finetuned_model.parameters()
)


print("Baseline epoch          :", BASELINE_EPOCH)
print(
    "Baseline validation loss:",
    round(BASELINE_VALIDATION_LOSS, 4),
)
print("Fine-tuned epoch        :", FINETUNED_EPOCH)
print(
    "Fine-tuned val loss     :",
    round(FINETUNED_VALIDATION_LOSS, 4),
)
print(
    "Pretrained epoch        :",
    finetuned_checkpoint["pretrained_epoch"],
)
print("Mapping checkpoint      : identik")
print("Mode model              : evaluation")
print("Gradient                : nonaktif")
print("✓ Cell 7 berhasil dijalankan.")

**Cell 8: Fungsi evaluasi dan perhitungan metrik**

In [ ]:
import unicodedata


def is_arabic_letter(char):
    if len(char) != 1:
        return False

    unicode_name = unicodedata.name(char, "")
    unicode_category = unicodedata.category(char)

    return (
        "ARABIC" in unicode_name
        and unicode_category.startswith("L")
    )


def reconstruct_diacritized_text(chars, label_ids):
    assert len(chars) == len(label_ids)

    output = []

    for char, label_id in zip(chars, label_ids):
        label = id2label[int(label_id)]
        output.append(char)

        if label != "O":
            output.extend(label.split("|"))

    return "".join(output)


def split_words(text):
    return re.findall(r"\S+", text)


def evaluate_model(model, model_name):
    model.eval()

    total_loss = 0.0
    total_labels = 0
    correct_labels = 0

    total_arabic_letters = 0
    correct_arabic_labels = 0

    total_words = 0
    incorrect_words = 0

    total_sequences = 0
    exact_sequences = 0

    all_true_label_ids = []
    all_predicted_label_ids = []
    prediction_rows = []

    progress_bar = tqdm(
        test_loader,
        desc=f"Evaluasi {model_name}",
        leave=False,
    )

    with torch.inference_mode():
        for batch in progress_bar:
            char_ids = batch["char_ids"].to(
                DEVICE,
                non_blocking=True,
            )

            label_ids = batch["label_ids"].to(
                DEVICE,
                non_blocking=True,
            )

            mask = batch["mask"].to(
                DEVICE,
                non_blocking=True,
            )

            lengths = batch["lengths"]

            emissions = model.compute_emissions(
                char_ids,
                lengths,
            )

            log_likelihood = model.crf(
                emissions,
                label_ids,
                mask=mask,
                reduction="token_mean",
            )

            loss = -log_likelihood

            decoded_batch = model.crf.decode(
                emissions,
                mask=mask,
            )

            valid_tokens = int(mask.sum().item())

            total_loss += loss.item() * valid_tokens

            for sample, predicted_ids in zip(
                batch["samples"],
                decoded_batch,
            ):
                gold_ids = sample["label_ids"]
                chars = sample["chars"]

                assert len(predicted_ids) == sample["length"]
                assert len(gold_ids) == sample["length"]
                assert len(chars) == sample["length"]

                predicted_text = (
                    reconstruct_diacritized_text(
                        chars,
                        predicted_ids,
                    )
                )

                gold_text = reconstruct_diacritized_text(
                    chars,
                    gold_ids,
                )

                assert gold_text == sample["text_diacritized"]

                sequence_label_errors = sum(
                    prediction != gold
                    for prediction, gold in zip(
                        predicted_ids,
                        gold_ids,
                    )
                )

                sequence_correct_labels = (
                    sample["length"]
                    - sequence_label_errors
                )

                arabic_positions = [
                    index
                    for index, char in enumerate(chars)
                    if is_arabic_letter(char)
                ]

                sequence_arabic_errors = sum(
                    predicted_ids[index]
                    != gold_ids[index]
                    for index in arabic_positions
                )

                sequence_correct_arabic = (
                    len(arabic_positions)
                    - sequence_arabic_errors
                )

                base_words = split_words(
                    sample["text_undiacritized"]
                )

                gold_words = split_words(gold_text)
                predicted_words = split_words(
                    predicted_text
                )

                assert (
                    len(base_words)
                    == len(gold_words)
                    == len(predicted_words)
                )

                sequence_total_words = 0
                sequence_incorrect_words = 0

                for base_word, gold_word, predicted_word in zip(
                    base_words,
                    gold_words,
                    predicted_words,
                ):
                    # Hanya token yang mengandung huruf Arab
                    if not any(
                        is_arabic_letter(char)
                        for char in base_word
                    ):
                        continue

                    sequence_total_words += 1

                    if predicted_word != gold_word:
                        sequence_incorrect_words += 1

                sequence_exact_match = (
                    sequence_label_errors == 0
                )

                total_labels += sample["length"]
                correct_labels += sequence_correct_labels

                total_arabic_letters += len(
                    arabic_positions
                )

                correct_arabic_labels += (
                    sequence_correct_arabic
                )

                total_words += sequence_total_words
                incorrect_words += (
                    sequence_incorrect_words
                )

                total_sequences += 1
                exact_sequences += int(
                    sequence_exact_match
                )

                all_true_label_ids.extend(gold_ids)
                all_predicted_label_ids.extend(
                    predicted_ids
                )

                prediction_rows.append({
                    "model": model_name,
                    "page": sample["page"],
                    "chunk_id": sample["chunk_id"],
                    "length": sample["length"],
                    "correct_labels": (
                        sequence_correct_labels
                    ),
                    "label_errors": (
                        sequence_label_errors
                    ),
                    "arabic_letters": len(
                        arabic_positions
                    ),
                    "arabic_errors": (
                        sequence_arabic_errors
                    ),
                    "total_words": (
                        sequence_total_words
                    ),
                    "word_errors": (
                        sequence_incorrect_words
                    ),
                    "exact_match": (
                        sequence_exact_match
                    ),
                    "text_undiacritized": (
                        sample["text_undiacritized"]
                    ),
                    "text_reference": gold_text,
                    "text_prediction": predicted_text,
                })

    assert total_sequences == len(test_dataset)
    assert total_labels == int(
        test_data["length"].sum()
    )

    accuracy = (
        correct_labels / total_labels * 100
    )

    der = (
        (
            total_arabic_letters
            - correct_arabic_labels
        )
        / total_arabic_letters
        * 100
    )

    arabic_accuracy = (
        correct_arabic_labels
        / total_arabic_letters
        * 100
    )

    wer = (
        incorrect_words / total_words * 100
    )

    exact_match = (
        exact_sequences / total_sequences * 100
    )

    metrics = {
        "model": model_name,
        "test_loss": total_loss / total_labels,
        "accuracy_percent": accuracy,
        "arabic_accuracy_percent": (
            arabic_accuracy
        ),
        "der_percent": der,
        "wer_percent": wer,
        "exact_match_percent": exact_match,
        "total_characters": total_labels,
        "correct_characters": correct_labels,
        "total_arabic_letters": (
            total_arabic_letters
        ),
        "arabic_label_errors": (
            total_arabic_letters
            - correct_arabic_labels
        ),
        "total_words": total_words,
        "incorrect_words": incorrect_words,
        "total_sequences": total_sequences,
        "exact_sequences": exact_sequences,
    }

    return {
        "metrics": metrics,
        "predictions": pd.DataFrame(
            prediction_rows
        ),
        "true_label_ids": all_true_label_ids,
        "predicted_label_ids": (
            all_predicted_label_ids
        ),
    }


# Memastikan rekonstruksi ground truth benar
sample_check = test_dataset[0]

reconstructed_check = reconstruct_diacritized_text(
    sample_check["chars"],
    sample_check["label_ids"],
)

assert (
    reconstructed_check
    == sample_check["text_diacritized"]
)

print("Rekonstruksi teks : valid")
print("Accuracy          : seluruh posisi karakter")
print("DER               : posisi huruf Arab")
print("WER               : token yang mengandung huruf Arab")
print("Exact Match       : seluruh sequence")
print("✓ Cell 8 berhasil dijalankan.")

**Cell 9: Evaluasi model baseline**

In [ ]:
baseline_result = evaluate_model(
    baseline_model,
    model_name="Baseline",
)

baseline_metrics = baseline_result["metrics"]
baseline_predictions = baseline_result["predictions"]

assert len(baseline_predictions) == len(test_dataset)
assert baseline_predictions["chunk_id"].is_unique

assert (
    len(baseline_result["true_label_ids"])
    == baseline_metrics["total_characters"]
)

assert (
    len(baseline_result["predicted_label_ids"])
    == baseline_metrics["total_characters"]
)

assert np.isfinite(
    baseline_metrics["test_loss"]
)

assert abs(
    baseline_metrics["arabic_accuracy_percent"]
    + baseline_metrics["der_percent"]
    - 100.0
) < 1e-8


baseline_predictions.to_csv(
    BASELINE_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)


baseline_summary = pd.DataFrame([
    {
        "Model": "Baseline",
        "Test Loss": baseline_metrics["test_loss"],
        "Accuracy (%)": (
            baseline_metrics["accuracy_percent"]
        ),
        "Arabic Accuracy (%)": (
            baseline_metrics[
                "arabic_accuracy_percent"
            ]
        ),
        "DER (%)": baseline_metrics["der_percent"],
        "WER (%)": baseline_metrics["wer_percent"],
        "Exact Match (%)": (
            baseline_metrics[
                "exact_match_percent"
            ]
        ),
    }
])

display(
    baseline_summary.style.format({
        "Test Loss": "{:.4f}",
        "Accuracy (%)": "{:.4f}",
        "Arabic Accuracy (%)": "{:.4f}",
        "DER (%)": "{:.4f}",
        "WER (%)": "{:.4f}",
        "Exact Match (%)": "{:.4f}",
    })
)


print(
    "Karakter benar       :",
    f'{baseline_metrics["correct_characters"]:,}',
    "dari",
    f'{baseline_metrics["total_characters"]:,}',
)

print(
    "Kesalahan huruf Arab :",
    f'{baseline_metrics["arabic_label_errors"]:,}',
    "dari",
    f'{baseline_metrics["total_arabic_letters"]:,}',
)

print(
    "Kata salah           :",
    f'{baseline_metrics["incorrect_words"]:,}',
    "dari",
    f'{baseline_metrics["total_words"]:,}',
)

print(
    "Sequence tepat       :",
    baseline_metrics["exact_sequences"],
    "dari",
    baseline_metrics["total_sequences"],
)

print(
    "Prediksi tersimpan   :",
    BASELINE_PREDICTIONS_PATH,
)

print("✓ Cell 9 berhasil dijalankan.")

**Cell 10: Evaluasi model fine-tuned**

In [ ]:
finetuned_result = evaluate_model(
    finetuned_model,
    model_name="Fine-tuned",
)

finetuned_metrics = finetuned_result["metrics"]
finetuned_predictions = (
    finetuned_result["predictions"]
)


assert len(finetuned_predictions) == len(
    test_dataset
)

assert finetuned_predictions[
    "chunk_id"
].is_unique

assert (
    finetuned_result["true_label_ids"]
    == baseline_result["true_label_ids"]
)

assert (
    finetuned_metrics["total_characters"]
    == baseline_metrics["total_characters"]
)

assert (
    finetuned_metrics["total_arabic_letters"]
    == baseline_metrics["total_arabic_letters"]
)

assert (
    finetuned_metrics["total_words"]
    == baseline_metrics["total_words"]
)

assert (
    finetuned_metrics["total_sequences"]
    == baseline_metrics["total_sequences"]
)

assert np.isfinite(
    finetuned_metrics["test_loss"]
)

assert abs(
    finetuned_metrics["arabic_accuracy_percent"]
    + finetuned_metrics["der_percent"]
    - 100.0
) < 1e-8


finetuned_predictions.to_csv(
    FINETUNED_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)


finetuned_summary = pd.DataFrame([
    {
        "Model": "Fine-tuned",
        "Test Loss": (
            finetuned_metrics["test_loss"]
        ),
        "Accuracy (%)": (
            finetuned_metrics[
                "accuracy_percent"
            ]
        ),
        "Arabic Accuracy (%)": (
            finetuned_metrics[
                "arabic_accuracy_percent"
            ]
        ),
        "DER (%)": (
            finetuned_metrics["der_percent"]
        ),
        "WER (%)": (
            finetuned_metrics["wer_percent"]
        ),
        "Exact Match (%)": (
            finetuned_metrics[
                "exact_match_percent"
            ]
        ),
    }
])

display(
    finetuned_summary.style.format({
        "Test Loss": "{:.4f}",
        "Accuracy (%)": "{:.4f}",
        "Arabic Accuracy (%)": "{:.4f}",
        "DER (%)": "{:.4f}",
        "WER (%)": "{:.4f}",
        "Exact Match (%)": "{:.4f}",
    })
)


print(
    "Karakter benar       :",
    f'{finetuned_metrics["correct_characters"]:,}',
    "dari",
    f'{finetuned_metrics["total_characters"]:,}',
)

print(
    "Kesalahan huruf Arab :",
    f'{finetuned_metrics["arabic_label_errors"]:,}',
    "dari",
    f'{finetuned_metrics["total_arabic_letters"]:,}',
)

print(
    "Kata salah           :",
    f'{finetuned_metrics["incorrect_words"]:,}',
    "dari",
    f'{finetuned_metrics["total_words"]:,}',
)

print(
    "Sequence tepat       :",
    finetuned_metrics["exact_sequences"],
    "dari",
    finetuned_metrics["total_sequences"],
)

print(
    "Prediksi tersimpan   :",
    FINETUNED_PREDICTIONS_PATH,
)

print("✓ Cell 10 berhasil dijalankan.")

**Cell 11: Perbandingan hasil kedua model**

In [ ]:
accuracy_gain_pp = (
    finetuned_metrics["accuracy_percent"]
    - baseline_metrics["accuracy_percent"]
)

arabic_accuracy_gain_pp = (
    finetuned_metrics["arabic_accuracy_percent"]
    - baseline_metrics["arabic_accuracy_percent"]
)

der_reduction_pp = (
    baseline_metrics["der_percent"]
    - finetuned_metrics["der_percent"]
)

der_relative_reduction = (
    der_reduction_pp
    / baseline_metrics["der_percent"]
    * 100
)

wer_reduction_pp = (
    baseline_metrics["wer_percent"]
    - finetuned_metrics["wer_percent"]
)

wer_relative_reduction = (
    wer_reduction_pp
    / baseline_metrics["wer_percent"]
    * 100
)

test_loss_reduction_percent = (
    (
        baseline_metrics["test_loss"]
        - finetuned_metrics["test_loss"]
    )
    / baseline_metrics["test_loss"]
    * 100
)

exact_match_gain_pp = (
    finetuned_metrics["exact_match_percent"]
    - baseline_metrics["exact_match_percent"]
)

character_errors_reduced = (
    (
        baseline_metrics["total_characters"]
        - baseline_metrics["correct_characters"]
    )
    - (
        finetuned_metrics["total_characters"]
        - finetuned_metrics["correct_characters"]
    )
)

arabic_errors_reduced = (
    baseline_metrics["arabic_label_errors"]
    - finetuned_metrics["arabic_label_errors"]
)

word_errors_reduced = (
    baseline_metrics["incorrect_words"]
    - finetuned_metrics["incorrect_words"]
)


metrics_comparison_df = pd.DataFrame([
    {
        **baseline_metrics,
        "checkpoint_epoch": BASELINE_EPOCH,
        "training_source": "Fathul Qorib",
    },
    {
        **finetuned_metrics,
        "checkpoint_epoch": FINETUNED_EPOCH,
        "training_source": (
            "Tashkeela -> Fathul Qorib"
        ),
    },
])


metrics_comparison_df.to_csv(
    METRICS_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)


improvement_summary = {
    "accuracy_gain_percentage_points": (
        accuracy_gain_pp
    ),
    "arabic_accuracy_gain_percentage_points": (
        arabic_accuracy_gain_pp
    ),
    "der_reduction_percentage_points": (
        der_reduction_pp
    ),
    "der_relative_reduction_percent": (
        der_relative_reduction
    ),
    "wer_reduction_percentage_points": (
        wer_reduction_pp
    ),
    "wer_relative_reduction_percent": (
        wer_relative_reduction
    ),
    "test_loss_relative_reduction_percent": (
        test_loss_reduction_percent
    ),
    "exact_match_gain_percentage_points": (
        exact_match_gain_pp
    ),
    "character_errors_reduced": (
        character_errors_reduced
    ),
    "arabic_errors_reduced": (
        arabic_errors_reduced
    ),
    "word_errors_reduced": (
        word_errors_reduced
    ),
}


evaluation_summary = {
    "test_set": {
        "pages": int(
            test_data["page"].nunique()
        ),
        "sequences": int(len(test_data)),
        "characters": int(
            baseline_metrics["total_characters"]
        ),
        "arabic_letters": int(
            baseline_metrics[
                "total_arabic_letters"
            ]
        ),
        "words": int(
            baseline_metrics["total_words"]
        ),
    },
    "baseline": baseline_metrics,
    "fine_tuned": finetuned_metrics,
    "improvement": improvement_summary,
}


with open(
    METRICS_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        evaluation_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


display_table = metrics_comparison_df[
    [
        "model",
        "checkpoint_epoch",
        "test_loss",
        "accuracy_percent",
        "arabic_accuracy_percent",
        "der_percent",
        "wer_percent",
        "exact_match_percent",
    ]
].copy()

display_table.columns = [
    "Model",
    "Epoch",
    "Test Loss",
    "Accuracy (%)",
    "Arabic Accuracy (%)",
    "DER (%)",
    "WER (%)",
    "Exact Match (%)",
]

display(
    display_table.style.format({
        "Test Loss": "{:.4f}",
        "Accuracy (%)": "{:.4f}",
        "Arabic Accuracy (%)": "{:.4f}",
        "DER (%)": "{:.4f}",
        "WER (%)": "{:.4f}",
        "Exact Match (%)": "{:.4f}",
    })
)


# Grafik perbandingan
METRICS_PLOT_PATH = (
    OUTPUT_DIR / "metrics_comparison.png"
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

model_names = ["Baseline", "Fine-tuned"]
colors = ["#6C757D", "#2E86DE"]


positive_metric_names = [
    "Accuracy",
    "Arabic Accuracy",
    "Exact Match",
]

baseline_positive = [
    baseline_metrics["accuracy_percent"],
    baseline_metrics["arabic_accuracy_percent"],
    baseline_metrics["exact_match_percent"],
]

finetuned_positive = [
    finetuned_metrics["accuracy_percent"],
    finetuned_metrics[
        "arabic_accuracy_percent"
    ],
    finetuned_metrics["exact_match_percent"],
]

x_positive = np.arange(
    len(positive_metric_names)
)

width = 0.35

axes[0].bar(
    x_positive - width / 2,
    baseline_positive,
    width,
    label=model_names[0],
    color=colors[0],
)

axes[0].bar(
    x_positive + width / 2,
    finetuned_positive,
    width,
    label=model_names[1],
    color=colors[1],
)

axes[0].set_title("Metrik Ketepatan")
axes[0].set_ylabel("Persentase (%)")
axes[0].set_xticks(x_positive)
axes[0].set_xticklabels(
    positive_metric_names,
    rotation=10,
)
axes[0].set_ylim(0, 105)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()


error_metric_names = ["DER", "WER"]

baseline_errors = [
    baseline_metrics["der_percent"],
    baseline_metrics["wer_percent"],
]

finetuned_errors = [
    finetuned_metrics["der_percent"],
    finetuned_metrics["wer_percent"],
]

x_errors = np.arange(
    len(error_metric_names)
)

axes[1].bar(
    x_errors - width / 2,
    baseline_errors,
    width,
    label=model_names[0],
    color=colors[0],
)

axes[1].bar(
    x_errors + width / 2,
    finetuned_errors,
    width,
    label=model_names[1],
    color=colors[1],
)

axes[1].set_title(
    "Metrik Kesalahan (lebih rendah lebih baik)"
)
axes[1].set_ylabel("Persentase (%)")
axes[1].set_xticks(x_errors)
axes[1].set_xticklabels(error_metric_names)
axes[1].set_ylim(
    0,
    max(baseline_errors) * 1.20,
)
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

plt.suptitle(
    "Perbandingan BiLSTM-CRF pada Test Set Fathul Qorib"
)

plt.tight_layout()

plt.savefig(
    METRICS_PLOT_PATH,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


print(
    "Kenaikan Accuracy       :",
    f"{accuracy_gain_pp:.4f} poin persentase",
)

print(
    "Penurunan relatif DER   :",
    f"{der_relative_reduction:.2f}%",
)

print(
    "Penurunan relatif WER   :",
    f"{wer_relative_reduction:.2f}%",
)

print(
    "Penurunan relatif loss  :",
    f"{test_loss_reduction_percent:.2f}%",
)

print(
    "Kesalahan karakter turun:",
    character_errors_reduced,
)

print(
    "Kesalahan Arab turun    :",
    arabic_errors_reduced,
)

print(
    "Kesalahan kata turun    :",
    word_errors_reduced,
)

print("Tabel CSV                :", METRICS_CSV_PATH)
print("Ringkasan JSON           :", METRICS_JSON_PATH)
print("Grafik                   :", METRICS_PLOT_PATH)
print("✓ Cell 11 berhasil dijalankan.")

**Cell 12: Evaluasi per label dan confusion matrix**

In [ ]:
LABEL_METRICS_BASELINE_PATH = (
    OUTPUT_DIR / "label_metrics_baseline.csv"
)

LABEL_METRICS_FINETUNED_PATH = (
    OUTPUT_DIR / "label_metrics_finetuned.csv"
)

CONFUSION_BASELINE_PATH = (
    OUTPUT_DIR / "confusion_matrix_baseline.png"
)

CONFUSION_FINETUNED_PATH = (
    OUTPUT_DIR / "confusion_matrix_finetuned.png"
)

CONFUSION_BASELINE_CSV_PATH = (
    OUTPUT_DIR / "confusion_matrix_baseline.csv"
)

CONFUSION_FINETUNED_CSV_PATH = (
    OUTPUT_DIR / "confusion_matrix_finetuned.csv"
)


MARK_NAMES = {
    "َ": "FATHA",
    "ُ": "DAMMA",
    "ِ": "KASRA",
    "ً": "FATHATAN",
    "ٌ": "DAMMATAN",
    "ٍ": "KASRATAN",
    "ّ": "SHADDA",
    "ْ": "SUKUN",
}


def readable_label(label):
    if label == "O":
        return "O"

    return "+".join(
        MARK_NAMES.get(mark, repr(mark))
        for mark in label.split("|")
    )


ordered_label_ids = sorted(id2label.keys())

ordered_label_names = [
    readable_label(id2label[label_id])
    for label_id in ordered_label_ids
]


def calculate_label_metrics(
    true_label_ids,
    predicted_label_ids,
    model_name,
):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            true_label_ids,
            predicted_label_ids,
            labels=ordered_label_ids,
            zero_division=0,
        )
    )

    confusion_counts = confusion_matrix(
        true_label_ids,
        predicted_label_ids,
        labels=ordered_label_ids,
    )

    confusion_normalized = confusion_matrix(
        true_label_ids,
        predicted_label_ids,
        labels=ordered_label_ids,
        normalize="true",
    ) * 100

    result = pd.DataFrame({
        "model": model_name,
        "label_id": ordered_label_ids,
        "label": [
            id2label[label_id]
            for label_id in ordered_label_ids
        ],
        "label_name": ordered_label_names,
        "precision_percent": precision * 100,
        "recall_percent": recall * 100,
        "f1_percent": f1 * 100,
        "support": support.astype(int),
        "correct_predictions": (
            np.diag(confusion_counts)
        ),
    })

    return (
        result,
        confusion_counts,
        confusion_normalized,
    )


(
    baseline_label_metrics,
    baseline_confusion_counts,
    baseline_confusion_normalized,
) = calculate_label_metrics(
    baseline_result["true_label_ids"],
    baseline_result["predicted_label_ids"],
    model_name="Baseline",
)


(
    finetuned_label_metrics,
    finetuned_confusion_counts,
    finetuned_confusion_normalized,
) = calculate_label_metrics(
    finetuned_result["true_label_ids"],
    finetuned_result["predicted_label_ids"],
    model_name="Fine-tuned",
)


assert baseline_label_metrics["support"].sum() == (
    baseline_metrics["total_characters"]
)

assert finetuned_label_metrics["support"].sum() == (
    finetuned_metrics["total_characters"]
)

assert baseline_label_metrics[
    "support"
].tolist() == finetuned_label_metrics[
    "support"
].tolist()


baseline_label_metrics.to_csv(
    LABEL_METRICS_BASELINE_PATH,
    index=False,
    encoding="utf-8-sig",
)

finetuned_label_metrics.to_csv(
    LABEL_METRICS_FINETUNED_PATH,
    index=False,
    encoding="utf-8-sig",
)


baseline_confusion_df = pd.DataFrame(
    baseline_confusion_counts,
    index=ordered_label_names,
    columns=ordered_label_names,
)

finetuned_confusion_df = pd.DataFrame(
    finetuned_confusion_counts,
    index=ordered_label_names,
    columns=ordered_label_names,
)

baseline_confusion_df.to_csv(
    CONFUSION_BASELINE_CSV_PATH,
    encoding="utf-8-sig",
)

finetuned_confusion_df.to_csv(
    CONFUSION_FINETUNED_CSV_PATH,
    encoding="utf-8-sig",
)


def plot_confusion_matrix(
    normalized_matrix,
    title,
    output_path,
):
    figure, axis = plt.subplots(
        figsize=(14, 12)
    )

    image = axis.imshow(
        normalized_matrix,
        interpolation="nearest",
        cmap="Blues",
        vmin=0,
        vmax=100,
    )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.04,
    )

    colorbar.set_label(
        "Persentase prediksi per label aktual (%)"
    )

    axis.set(
        xticks=np.arange(
            len(ordered_label_names)
        ),
        yticks=np.arange(
            len(ordered_label_names)
        ),
        xticklabels=ordered_label_names,
        yticklabels=ordered_label_names,
        xlabel="Label Prediksi",
        ylabel="Label Aktual",
        title=title,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    for row_index in range(
        normalized_matrix.shape[0]
    ):
        for column_index in range(
            normalized_matrix.shape[1]
        ):
            value = normalized_matrix[
                row_index,
                column_index,
            ]

            # Nilai sangat kecil tidak ditulis agar terbaca
            if value >= 1.0:
                text_color = (
                    "white"
                    if value >= 50
                    else "black"
                )

                axis.text(
                    column_index,
                    row_index,
                    f"{value:.1f}",
                    ha="center",
                    va="center",
                    color=text_color,
                    fontsize=7,
                )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)


plot_confusion_matrix(
    baseline_confusion_normalized,
    (
        "Confusion Matrix Baseline "
        "(Normalisasi per Label Aktual)"
    ),
    CONFUSION_BASELINE_PATH,
)

plot_confusion_matrix(
    finetuned_confusion_normalized,
    (
        "Confusion Matrix Fine-tuned "
        "(Normalisasi per Label Aktual)"
    ),
    CONFUSION_FINETUNED_PATH,
)


label_comparison = (
    baseline_label_metrics[
        [
            "label_id",
            "label_name",
            "support",
            "precision_percent",
            "recall_percent",
            "f1_percent",
        ]
    ]
    .rename(columns={
        "precision_percent": (
            "baseline_precision_percent"
        ),
        "recall_percent": (
            "baseline_recall_percent"
        ),
        "f1_percent": "baseline_f1_percent",
    })
    .merge(
        finetuned_label_metrics[
            [
                "label_id",
                "precision_percent",
                "recall_percent",
                "f1_percent",
            ]
        ].rename(columns={
            "precision_percent": (
                "finetuned_precision_percent"
            ),
            "recall_percent": (
                "finetuned_recall_percent"
            ),
            "f1_percent": (
                "finetuned_f1_percent"
            ),
        }),
        on="label_id",
        validate="one_to_one",
    )
)

label_comparison["f1_gain_pp"] = (
    label_comparison["finetuned_f1_percent"]
    - label_comparison["baseline_f1_percent"]
)


display(
    label_comparison[
        [
            "label_name",
            "support",
            "baseline_precision_percent",
            "baseline_recall_percent",
            "baseline_f1_percent",
            "finetuned_precision_percent",
            "finetuned_recall_percent",
            "finetuned_f1_percent",
            "f1_gain_pp",
        ]
    ].style.format({
        "baseline_precision_percent": "{:.2f}",
        "baseline_recall_percent": "{:.2f}",
        "baseline_f1_percent": "{:.2f}",
        "finetuned_precision_percent": "{:.2f}",
        "finetuned_recall_percent": "{:.2f}",
        "finetuned_f1_percent": "{:.2f}",
        "f1_gain_pp": "{:+.2f}",
    })
)


baseline_macro_f1 = float(
    baseline_label_metrics[
        "f1_percent"
    ].mean()
)

finetuned_macro_f1 = float(
    finetuned_label_metrics[
        "f1_percent"
    ].mean()
)

macro_f1_gain_pp = (
    finetuned_macro_f1
    - baseline_macro_f1
)


print(
    "Macro F1 baseline   :",
    f"{baseline_macro_f1:.4f}%",
)

print(
    "Macro F1 fine-tuned :",
    f"{finetuned_macro_f1:.4f}%",
)

print(
    "Kenaikan Macro F1   :",
    f"{macro_f1_gain_pp:+.4f} poin persentase",
)

print(
    "Label membaik       :",
    int(
        (
            label_comparison["f1_gain_pp"]
            > 0
        ).sum()
    ),
    "dari",
    len(label_comparison),
)

print(
    "Label menurun       :",
    int(
        (
            label_comparison["f1_gain_pp"]
            < 0
        ).sum()
    ),
    "dari",
    len(label_comparison),
)

print(
    "Metrik baseline     :",
    LABEL_METRICS_BASELINE_PATH,
)

print(
    "Metrik fine-tuned   :",
    LABEL_METRICS_FINETUNED_PATH,
)

print(
    "Confusion baseline  :",
    CONFUSION_BASELINE_PATH,
)

print(
    "Confusion fine-tuned:",
    CONFUSION_FINETUNED_PATH,
)

print("✓ Cell 12 berhasil dijalankan.")

**Cell 13: Analisis kesalahan per sequence dan halaman**

In [ ]:
ERROR_ANALYSIS_PATH = (
    OUTPUT_DIR / "error_analysis.csv"
)

PAGE_ERROR_ANALYSIS_PATH = (
    OUTPUT_DIR / "page_error_analysis.csv"
)


baseline_error_data = baseline_predictions[
    [
        "page",
        "chunk_id",
        "length",
        "text_undiacritized",
        "text_reference",
        "text_prediction",
        "label_errors",
        "arabic_letters",
        "arabic_errors",
        "total_words",
        "word_errors",
        "exact_match",
    ]
].rename(columns={
    "text_prediction": "prediction_baseline",
    "label_errors": "label_errors_baseline",
    "arabic_letters": "arabic_letters_baseline",
    "arabic_errors": "arabic_errors_baseline",
    "total_words": "total_words_baseline",
    "word_errors": "word_errors_baseline",
    "exact_match": "exact_match_baseline",
})


finetuned_error_data = finetuned_predictions[
    [
        "page",
        "chunk_id",
        "length",
        "text_undiacritized",
        "text_reference",
        "text_prediction",
        "label_errors",
        "arabic_letters",
        "arabic_errors",
        "total_words",
        "word_errors",
        "exact_match",
    ]
].rename(columns={
    "page": "page_finetuned",
    "length": "length_finetuned",
    "text_undiacritized": (
        "text_undiacritized_finetuned"
    ),
    "text_reference": "text_reference_finetuned",
    "text_prediction": "prediction_finetuned",
    "label_errors": "label_errors_finetuned",
    "arabic_letters": "arabic_letters_finetuned",
    "arabic_errors": "arabic_errors_finetuned",
    "total_words": "total_words_finetuned",
    "word_errors": "word_errors_finetuned",
    "exact_match": "exact_match_finetuned",
})


error_analysis = baseline_error_data.merge(
    finetuned_error_data,
    on="chunk_id",
    how="inner",
    validate="one_to_one",
)


assert len(error_analysis) == len(test_dataset)

assert (
    error_analysis["page"]
    == error_analysis["page_finetuned"]
).all()

assert (
    error_analysis["length"]
    == error_analysis["length_finetuned"]
).all()

assert (
    error_analysis["text_undiacritized"]
    == error_analysis[
        "text_undiacritized_finetuned"
    ]
).all()

assert (
    error_analysis["text_reference"]
    == error_analysis[
        "text_reference_finetuned"
    ]
).all()

assert (
    error_analysis["arabic_letters_baseline"]
    == error_analysis["arabic_letters_finetuned"]
).all()

assert (
    error_analysis["total_words_baseline"]
    == error_analysis["total_words_finetuned"]
).all()


# Menghapus kolom validasi yang sudah dipastikan sama
error_analysis = error_analysis.drop(columns=[
    "page_finetuned",
    "length_finetuned",
    "text_undiacritized_finetuned",
    "text_reference_finetuned",
    "arabic_letters_finetuned",
    "total_words_finetuned",
])


error_analysis["der_baseline_percent"] = (
    error_analysis["arabic_errors_baseline"]
    / error_analysis["arabic_letters_baseline"]
    * 100
)

error_analysis["der_finetuned_percent"] = (
    error_analysis["arabic_errors_finetuned"]
    / error_analysis["arabic_letters_baseline"]
    * 100
)

error_analysis["wer_baseline_percent"] = (
    error_analysis["word_errors_baseline"]
    / error_analysis["total_words_baseline"]
    * 100
)

error_analysis["wer_finetuned_percent"] = (
    error_analysis["word_errors_finetuned"]
    / error_analysis["total_words_baseline"]
    * 100
)

error_analysis["label_errors_reduction"] = (
    error_analysis["label_errors_baseline"]
    - error_analysis["label_errors_finetuned"]
)

error_analysis["arabic_errors_reduction"] = (
    error_analysis["arabic_errors_baseline"]
    - error_analysis["arabic_errors_finetuned"]
)

error_analysis["word_errors_reduction"] = (
    error_analysis["word_errors_baseline"]
    - error_analysis["word_errors_finetuned"]
)


error_analysis["sequence_category"] = np.select(
    [
        (
            error_analysis["arabic_errors_finetuned"]
            < error_analysis["arabic_errors_baseline"]
        ),
        (
            error_analysis["arabic_errors_finetuned"]
            > error_analysis["arabic_errors_baseline"]
        ),
    ],
    [
        "membaik",
        "menurun",
    ],
    default="tetap",
)


error_analysis = error_analysis.sort_values(
    [
        "der_finetuned_percent",
        "word_errors_finetuned",
    ],
    ascending=[False, False],
).reset_index(drop=True)


error_analysis.to_csv(
    ERROR_ANALYSIS_PATH,
    index=False,
    encoding="utf-8-sig",
)


# Agregasi kesalahan berdasarkan halaman
page_error_analysis = (
    error_analysis.groupby(
        "page",
        as_index=False,
    )
    .agg(
        sequences=("chunk_id", "count"),
        characters=("length", "sum"),
        arabic_letters=(
            "arabic_letters_baseline",
            "sum",
        ),
        words=("total_words_baseline", "sum"),
        label_errors_baseline=(
            "label_errors_baseline",
            "sum",
        ),
        label_errors_finetuned=(
            "label_errors_finetuned",
            "sum",
        ),
        arabic_errors_baseline=(
            "arabic_errors_baseline",
            "sum",
        ),
        arabic_errors_finetuned=(
            "arabic_errors_finetuned",
            "sum",
        ),
        word_errors_baseline=(
            "word_errors_baseline",
            "sum",
        ),
        word_errors_finetuned=(
            "word_errors_finetuned",
            "sum",
        ),
    )
)


page_error_analysis["der_baseline_percent"] = (
    page_error_analysis["arabic_errors_baseline"]
    / page_error_analysis["arabic_letters"]
    * 100
)

page_error_analysis["der_finetuned_percent"] = (
    page_error_analysis["arabic_errors_finetuned"]
    / page_error_analysis["arabic_letters"]
    * 100
)

page_error_analysis["wer_baseline_percent"] = (
    page_error_analysis["word_errors_baseline"]
    / page_error_analysis["words"]
    * 100
)

page_error_analysis["wer_finetuned_percent"] = (
    page_error_analysis["word_errors_finetuned"]
    / page_error_analysis["words"]
    * 100
)

page_error_analysis["der_reduction_pp"] = (
    page_error_analysis["der_baseline_percent"]
    - page_error_analysis["der_finetuned_percent"]
)

page_error_analysis["wer_reduction_pp"] = (
    page_error_analysis["wer_baseline_percent"]
    - page_error_analysis["wer_finetuned_percent"]
)


page_error_analysis = page_error_analysis.sort_values(
    "der_finetuned_percent",
    ascending=False,
).reset_index(drop=True)


page_error_analysis.to_csv(
    PAGE_ERROR_ANALYSIS_PATH,
    index=False,
    encoding="utf-8-sig",
)


category_summary = (
    error_analysis["sequence_category"]
    .value_counts()
    .rename_axis("Kategori")
    .reset_index(name="Jumlah sequence")
)

display(category_summary)


print("\n10 sequence dengan DER fine-tuned tertinggi:")

display(
    error_analysis[
        [
            "page",
            "chunk_id",
            "arabic_letters_baseline",
            "arabic_errors_baseline",
            "arabic_errors_finetuned",
            "der_baseline_percent",
            "der_finetuned_percent",
            "word_errors_baseline",
            "word_errors_finetuned",
            "sequence_category",
        ]
    ]
    .head(10)
    .style.format({
        "der_baseline_percent": "{:.2f}",
        "der_finetuned_percent": "{:.2f}",
    })
)


largest_improvements = (
    error_analysis.sort_values(
        [
            "arabic_errors_reduction",
            "word_errors_reduction",
        ],
        ascending=[False, False],
    )
    .head(10)
)

print("\n10 sequence dengan penurunan kesalahan terbesar:")

display(
    largest_improvements[
        [
            "page",
            "chunk_id",
            "arabic_errors_baseline",
            "arabic_errors_finetuned",
            "arabic_errors_reduction",
            "word_errors_baseline",
            "word_errors_finetuned",
            "word_errors_reduction",
        ]
    ]
)


print("\n10 halaman dengan DER fine-tuned tertinggi:")

display(
    page_error_analysis[
        [
            "page",
            "sequences",
            "arabic_letters",
            "der_baseline_percent",
            "der_finetuned_percent",
            "der_reduction_pp",
            "wer_baseline_percent",
            "wer_finetuned_percent",
            "wer_reduction_pp",
        ]
    ]
    .head(10)
    .style.format({
        "der_baseline_percent": "{:.2f}",
        "der_finetuned_percent": "{:.2f}",
        "der_reduction_pp": "{:+.2f}",
        "wer_baseline_percent": "{:.2f}",
        "wer_finetuned_percent": "{:.2f}",
        "wer_reduction_pp": "{:+.2f}",
    })
)


# Contoh teks untuk analisis kualitatif
qualitative_examples = largest_improvements[
    [
        "page",
        "chunk_id",
        "text_undiacritized",
        "text_reference",
        "prediction_baseline",
        "prediction_finetuned",
        "arabic_errors_reduction",
        "word_errors_reduction",
    ]
].head(5)

print("\nContoh perbandingan prediksi:")

with pd.option_context(
    "display.max_colwidth",
    180,
):
    display(qualitative_examples)


improved_sequences = int(
    (
        error_analysis["sequence_category"]
        == "membaik"
    ).sum()
)

worsened_sequences = int(
    (
        error_analysis["sequence_category"]
        == "menurun"
    ).sum()
)

unchanged_sequences = int(
    (
        error_analysis["sequence_category"]
        == "tetap"
    ).sum()
)


print("\nSequence membaik :", improved_sequences)
print("Sequence menurun :", worsened_sequences)
print("Sequence tetap   :", unchanged_sequences)

print(
    "Rata-rata DER baseline   :",
    f'{baseline_metrics["der_percent"]:.4f}%',
)

print(
    "Rata-rata DER fine-tuned :",
    f'{finetuned_metrics["der_percent"]:.4f}%',
)

print("Analisis sequence :", ERROR_ANALYSIS_PATH)
print("Analisis halaman  :", PAGE_ERROR_ANALYSIS_PATH)
print("✓ Cell 13 berhasil dijalankan.")

**Cell 14: Audit dan validasi akhir evaluasi**

In [ ]:
EVALUATION_AUDIT_PATH = (
    OUTPUT_DIR / "evaluation_audit.json"
)


# Validasi protokol evaluasi
assert BASELINE_EPOCH == 78
assert FINETUNED_EPOCH == 46

assert baseline_checkpoint["char2id"] == char2id
assert baseline_checkpoint["label2id"] == label2id

assert finetuned_checkpoint["char2id"] == char2id
assert finetuned_checkpoint["label2id"] == label2id

assert (
    baseline_result["true_label_ids"]
    == finetuned_result["true_label_ids"]
)

assert len(
    baseline_result["predicted_label_ids"]
) == int(test_data["length"].sum())

assert len(
    finetuned_result["predicted_label_ids"]
) == int(test_data["length"].sum())

assert not any(
    parameter.requires_grad
    for parameter in baseline_model.parameters()
)

assert not any(
    parameter.requires_grad
    for parameter in finetuned_model.parameters()
)


# Validasi konsistensi hasil agregat
assert int(
    error_analysis["label_errors_baseline"].sum()
) == (
    baseline_metrics["total_characters"]
    - baseline_metrics["correct_characters"]
)

assert int(
    error_analysis["label_errors_finetuned"].sum()
) == (
    finetuned_metrics["total_characters"]
    - finetuned_metrics["correct_characters"]
)

assert int(
    error_analysis["arabic_errors_baseline"].sum()
) == baseline_metrics["arabic_label_errors"]

assert int(
    error_analysis["arabic_errors_finetuned"].sum()
) == finetuned_metrics["arabic_label_errors"]

assert int(
    error_analysis["word_errors_baseline"].sum()
) == baseline_metrics["incorrect_words"]

assert int(
    error_analysis["word_errors_finetuned"].sum()
) == finetuned_metrics["incorrect_words"]


# Validasi hasil utama
assert (
    finetuned_metrics["test_loss"]
    < baseline_metrics["test_loss"]
)

assert (
    finetuned_metrics["accuracy_percent"]
    > baseline_metrics["accuracy_percent"]
)

assert (
    finetuned_metrics["der_percent"]
    < baseline_metrics["der_percent"]
)

assert (
    finetuned_metrics["wer_percent"]
    < baseline_metrics["wer_percent"]
)


labels_improved = int(
    (label_comparison["f1_gain_pp"] > 0).sum()
)

labels_declined = int(
    (label_comparison["f1_gain_pp"] < 0).sum()
)

labels_unchanged = int(
    (label_comparison["f1_gain_pp"] == 0).sum()
)


evaluation_audit = {
    "status": "passed",
    "evaluation_protocol": {
        "test_set_used_for_training": False,
        "model_selection_used_test_set": False,
        "same_test_set_for_both_models": True,
        "same_architecture": True,
        "same_vocabulary_mapping": True,
        "same_label_mapping": True,
        "gradient_disabled": True,
        "crf_viterbi_decoding": True,
    },
    "test_set": {
        "pages": int(
            test_data["page"].nunique()
        ),
        "sequences": int(len(test_data)),
        "characters": int(
            baseline_metrics["total_characters"]
        ),
        "arabic_letters": int(
            baseline_metrics[
                "total_arabic_letters"
            ]
        ),
        "arabic_words": int(
            baseline_metrics["total_words"]
        ),
        "vocabulary_size": int(len(char2id)),
        "number_of_labels": int(len(label2id)),
        "unknown_characters": dict(
            test_dataset.unknown_characters
        ),
    },
    "baseline": {
        "training_source": "Fathul Qorib",
        "checkpoint_epoch": int(BASELINE_EPOCH),
        "validation_loss": float(
            BASELINE_VALIDATION_LOSS
        ),
        "test_loss": float(
            baseline_metrics["test_loss"]
        ),
        "accuracy_percent": float(
            baseline_metrics["accuracy_percent"]
        ),
        "arabic_accuracy_percent": float(
            baseline_metrics[
                "arabic_accuracy_percent"
            ]
        ),
        "der_percent": float(
            baseline_metrics["der_percent"]
        ),
        "wer_percent": float(
            baseline_metrics["wer_percent"]
        ),
        "exact_match_percent": float(
            baseline_metrics[
                "exact_match_percent"
            ]
        ),
        "macro_f1_percent": float(
            baseline_macro_f1
        ),
    },
    "fine_tuned": {
        "training_source": (
            "Tashkeela -> Fathul Qorib"
        ),
        "pretrained_epoch": int(
            finetuned_checkpoint[
                "pretrained_epoch"
            ]
        ),
        "checkpoint_epoch": int(
            FINETUNED_EPOCH
        ),
        "validation_loss": float(
            FINETUNED_VALIDATION_LOSS
        ),
        "test_loss": float(
            finetuned_metrics["test_loss"]
        ),
        "accuracy_percent": float(
            finetuned_metrics[
                "accuracy_percent"
            ]
        ),
        "arabic_accuracy_percent": float(
            finetuned_metrics[
                "arabic_accuracy_percent"
            ]
        ),
        "der_percent": float(
            finetuned_metrics["der_percent"]
        ),
        "wer_percent": float(
            finetuned_metrics["wer_percent"]
        ),
        "exact_match_percent": float(
            finetuned_metrics[
                "exact_match_percent"
            ]
        ),
        "macro_f1_percent": float(
            finetuned_macro_f1
        ),
    },
    "improvement": {
        "accuracy_gain_percentage_points": float(
            accuracy_gain_pp
        ),
        "arabic_accuracy_gain_percentage_points": (
            float(arabic_accuracy_gain_pp)
        ),
        "der_reduction_percentage_points": float(
            der_reduction_pp
        ),
        "der_relative_reduction_percent": float(
            der_relative_reduction
        ),
        "wer_reduction_percentage_points": float(
            wer_reduction_pp
        ),
        "wer_relative_reduction_percent": float(
            wer_relative_reduction
        ),
        "test_loss_relative_reduction_percent": (
            float(test_loss_reduction_percent)
        ),
        "exact_match_gain_percentage_points": (
            float(exact_match_gain_pp)
        ),
        "macro_f1_gain_percentage_points": float(
            macro_f1_gain_pp
        ),
        "character_errors_reduced": int(
            character_errors_reduced
        ),
        "arabic_errors_reduced": int(
            arabic_errors_reduced
        ),
        "word_errors_reduced": int(
            word_errors_reduced
        ),
    },
    "label_analysis": {
        "labels_total": int(len(label2id)),
        "labels_improved": labels_improved,
        "labels_declined": labels_declined,
        "labels_unchanged": labels_unchanged,
    },
    "sequence_analysis": {
        "sequences_improved": int(
            improved_sequences
        ),
        "sequences_declined": int(
            worsened_sequences
        ),
        "sequences_unchanged": int(
            unchanged_sequences
        ),
    },
}


with open(
    EVALUATION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        evaluation_audit,
        file,
        ensure_ascii=False,
        indent=2,
    )


evaluation_outputs = [
    METRICS_CSV_PATH,
    METRICS_JSON_PATH,
    BASELINE_PREDICTIONS_PATH,
    FINETUNED_PREDICTIONS_PATH,
    LABEL_METRICS_BASELINE_PATH,
    LABEL_METRICS_FINETUNED_PATH,
    CONFUSION_BASELINE_PATH,
    CONFUSION_FINETUNED_PATH,
    CONFUSION_BASELINE_CSV_PATH,
    CONFUSION_FINETUNED_CSV_PATH,
    METRICS_PLOT_PATH,
    ERROR_ANALYSIS_PATH,
    PAGE_ERROR_ANALYSIS_PATH,
    EVALUATION_AUDIT_PATH,
]


for file_path in evaluation_outputs:
    assert file_path.exists(), (
        f"Output tidak ditemukan: {file_path}"
    )

    assert file_path.stat().st_size > 0, (
        f"Output kosong: {file_path}"
    )


print("Status audit              :", evaluation_audit["status"])
print("Test digunakan training   : Tidak")
print("Pemilihan model dari test : Tidak")
print("Test set kedua model sama : Ya")
print("Arsitektur kedua model    : Sama")
print("Mapping kedua model       : Sama")

print("\nHASIL UTAMA TEST SET")
print(
    "Baseline Accuracy         :",
    f'{baseline_metrics["accuracy_percent"]:.4f}%',
)
print(
    "Fine-tuned Accuracy       :",
    f'{finetuned_metrics["accuracy_percent"]:.4f}%',
)
print(
    "Baseline DER              :",
    f'{baseline_metrics["der_percent"]:.4f}%',
)
print(
    "Fine-tuned DER            :",
    f'{finetuned_metrics["der_percent"]:.4f}%',
)
print(
    "Penurunan relatif DER     :",
    f"{der_relative_reduction:.2f}%",
)
print(
    "Baseline WER              :",
    f'{baseline_metrics["wer_percent"]:.4f}%',
)
print(
    "Fine-tuned WER            :",
    f'{finetuned_metrics["wer_percent"]:.4f}%',
)
print(
    "Penurunan relatif WER     :",
    f"{wer_relative_reduction:.2f}%",
)

print("\nJumlah output tervalidasi :", len(evaluation_outputs))
print("Folder evaluasi           :", OUTPUT_DIR)
print("Audit evaluasi             :", EVALUATION_AUDIT_PATH)
print("✓ Notebook 05 selesai dan tervalidasi.")